# English Novel Psych Verbs — LLM Testing
## High-Resource Language Replication: *norp* and *frum*

**Purpose:** Tests whether LLMs fail on the habitual/episodic paradigm in English (high-resource),
to determine if the Uzbek failure was due to low-resource encoding or a genuine limit of distributional learning.

**Key methodological note:** English has no overt nominal case morphology.
Both answer choices are bare NPs (e.g. 'Puppy', 'Monkey'), so this test is
inherently equivalent to the no-overt-case-marker condition in Uzbek.
There is no morphological confound — the only cue available to models is the
semantic-aspectual framing of the story.

**Models tested:** mBERT, XLM-RoBERTa-base, XLM-RoBERTa-large, XLM-V-base
(BERTbek excluded — Uzbek-only model)

**Instructions:**
1. Upload `english_psych_verbs_32items.json` when prompted
2. Click Runtime → Run All
3. Wait ~60 minutes
4. Download results at the end

In [ ]:
# STEP 1: Install packages
!pip install transformers torch pandas matplotlib seaborn scipy -q
print('✅ Packages installed!')

In [ ]:
# STEP 2: Upload dataset
from google.colab import files
import json

print('Please upload english_psych_verbs_32items.json:')
uploaded = files.upload()

with open('english_psych_verbs_32items.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

print(f'\n✅ Loaded {len(dataset)} items')
print(f'   ES condition: {sum(1 for i in dataset if i["condition"]=="ES")} items')
print(f'   EO condition: {sum(1 for i in dataset if i["condition"]=="EO")} items')
print(f'   norp: {sum(1 for i in dataset if i["verb"]=="norp")} items')
print(f'   frum: {sum(1 for i in dataset if i["verb"]=="frum")} items')
print(f'\n⚠️  Note: Both answer choices are bare NPs — no morphological cue available.')
print(f'   This is inherently equivalent to the no-overt-case-marker Uzbek condition.')

In [ ]:
# STEP 3: Define testing functions
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForMaskedLM

def calculate_perplexity(model, tokenizer, text):
    """Calculate perplexity for a text sequence."""
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs['input_ids'])
        loss = outputs.loss.item()
    return np.exp(loss)

def test_item(item, model, tokenizer):
    """Test single item. Both answers are bare NPs — no morphological cue."""
    context = f"{item['verb_definition']}\n\n{item['story_part1']}\n\n{item['story_part2']}\n\n"
    question = item['test_question']

    answer1 = item['answer_choices']['answer_1']
    answer2 = item['answer_choices']['answer_2']

    text1 = f"{context}{question} {answer1}"
    text2 = f"{context}{question} {answer2}"

    ppl1 = calculate_perplexity(model, tokenizer, text1)
    ppl2 = calculate_perplexity(model, tokenizer, text2)

    # EO preference = 1 if model prefers answer_2 (the EO-structure animal)
    eo_pref = 1 if ppl2 < ppl1 else 0

    return {
        'item_id': item['id'],
        'condition': item['condition'],
        'verb': item['verb'],
        'ppl_answer1': ppl1,
        'ppl_answer2': ppl2,
        'eo_preference': eo_pref
    }

print('✅ Testing function defined!')

In [ ]:
# STEP 4: Test mBERT
import pandas as pd

print('🔄 Loading mBERT (180M params)...')
tokenizer_mbert = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')
model_mbert = AutoModelForMaskedLM.from_pretrained('bert-base-multilingual-cased')
model_mbert.eval()
print('✅ mBERT loaded!\n')

results_mbert = []
for i, item in enumerate(dataset, 1):
    print(f'  {i}/32: {item["id"][:25]}...', end=' ')
    result = test_item(item, model_mbert, tokenizer_mbert)
    results_mbert.append(result)
    print(f'✓ (EO={result["eo_preference"]})')

df_mbert = pd.DataFrame(results_mbert)
print('\n✅ mBERT complete!')

In [ ]:
# STEP 5: Test XLM-RoBERTa-base
print('🔄 Loading XLM-RoBERTa-base (270M params)...')
tokenizer_base = AutoTokenizer.from_pretrained('xlm-roberta-base')
model_base = AutoModelForMaskedLM.from_pretrained('xlm-roberta-base')
model_base.eval()
print('✅ XLM-RoBERTa-base loaded!\n')

results_base = []
for i, item in enumerate(dataset, 1):
    print(f'  {i}/32: {item["id"][:25]}...', end=' ')
    result = test_item(item, model_base, tokenizer_base)
    results_base.append(result)
    print(f'✓ (EO={result["eo_preference"]})')

df_base = pd.DataFrame(results_base)
print('\n✅ XLM-RoBERTa-base complete!')

In [ ]:
# STEP 6: Test XLM-RoBERTa-large
print('🔄 Loading XLM-RoBERTa-large (550M params)...')
tokenizer_large = AutoTokenizer.from_pretrained('xlm-roberta-large')
model_large = AutoModelForMaskedLM.from_pretrained('xlm-roberta-large')
model_large.eval()
print('✅ XLM-RoBERTa-large loaded!\n')

results_large = []
for i, item in enumerate(dataset, 1):
    print(f'  {i}/32: {item["id"][:25]}...', end=' ')
    result = test_item(item, model_large, tokenizer_large)
    results_large.append(result)
    print(f'✓ (EO={result["eo_preference"]})')

df_large = pd.DataFrame(results_large)
print('\n✅ XLM-RoBERTa-large complete!')

In [ ]:
# STEP 7: Test XLM-V-base
print('🔄 Loading XLM-V-base (1.2B params)...')
tokenizer_xlmv = AutoTokenizer.from_pretrained('facebook/xlm-v-base')
model_xlmv = AutoModelForMaskedLM.from_pretrained('facebook/xlm-v-base')
model_xlmv.eval()
print('✅ XLM-V-base loaded!\n')

results_xlmv = []
for i, item in enumerate(dataset, 1):
    print(f'  {i}/32: {item["id"][:25]}...', end=' ')
    result = test_item(item, model_xlmv, tokenizer_xlmv)
    results_xlmv.append(result)
    print(f'✓ (EO={result["eo_preference"]})')

df_xlmv = pd.DataFrame(results_xlmv)
print('\n✅ XLM-V-base complete!')

In [ ]:
# STEP 8: Calculate and display results
print('\n' + '='*70)
print('📊 FINAL RESULTS — ENGLISH (No Overt Case Marker by Design)')
print('='*70)

mbert_es   = df_mbert[df_mbert['condition']=='ES']['eo_preference'].mean() * 100
mbert_eo   = df_mbert[df_mbert['condition']=='EO']['eo_preference'].mean() * 100
base_es    = df_base[df_base['condition']=='ES']['eo_preference'].mean() * 100
base_eo    = df_base[df_base['condition']=='EO']['eo_preference'].mean() * 100
large_es   = df_large[df_large['condition']=='ES']['eo_preference'].mean() * 100
large_eo   = df_large[df_large['condition']=='EO']['eo_preference'].mean() * 100
xlmv_es    = df_xlmv[df_xlmv['condition']=='ES']['eo_preference'].mean() * 100
xlmv_eo    = df_xlmv[df_xlmv['condition']=='EO']['eo_preference'].mean() * 100

child_es, child_eo = 36.0, 76.0  # Uzbek children baseline

print(f"\n{'Model':<30} {'ES Pref':<12} {'EO Pref':<12} {'Difference'}")
print('-' * 65)
print(f"{'Children (Uzbek, N=40)':<30} {child_es:>6.1f}%     {child_eo:>6.1f}%     {child_eo-child_es:+.1f} pp")
print(f"{'mBERT (180M)':<30} {mbert_es:>6.1f}%     {mbert_eo:>6.1f}%     {mbert_eo-mbert_es:+.1f} pp")
print(f"{'XLM-RoBERTa-base (270M)':<30} {base_es:>6.1f}%     {base_eo:>6.1f}%     {base_eo-base_es:+.1f} pp")
print(f"{'XLM-RoBERTa-large (550M)':<30} {large_es:>6.1f}%     {large_eo:>6.1f}%     {large_eo-large_es:+.1f} pp")
print(f"{'XLM-V-base (1.2B)':<30} {xlmv_es:>6.1f}%     {xlmv_eo:>6.1f}%     {xlmv_eo-xlmv_es:+.1f} pp")

print('\n' + '='*70)
print('INTERPRETATION GUIDE:')
print('  If all differences ≈ 0  → Models fail in English too.')
print('     Conclusion: distributional learning is insufficient regardless of language.')
print('  If differences > 20pp  → Models succeed in English.')
print('     Conclusion: Uzbek low-resource encoding was the primary obstacle.')
print('='*70)

In [ ]:
# STEP 9: Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

models  = ['Children\n(Uzbek,N=40)', 'mBERT\n(180M)', 'XLM-R-base\n(270M)', 'XLM-R-large\n(550M)', 'XLM-V-base\n(1.2B)']
es_vals = [child_es, mbert_es, base_es, large_es, xlmv_es]
eo_vals = [child_eo, mbert_eo, base_eo, large_eo, xlmv_eo]

import numpy as np
x = np.arange(len(models))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - w/2, es_vals, w, color='#E8835A', label='ES (habitual)')
ax.bar(x + w/2, eo_vals, w, color='#2A7F7F', label='EO (episodic)')
ax.axhline(50, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='Chance (50%)')
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=10)
ax.set_ylabel('EO Preference (%)', fontsize=12)
ax.set_ylim(0, 100)
ax.set_title('English Novel Psych Verbs (norp + frum)\nArgument Linking Preferences — No Overt Case Marker', fontsize=13, fontweight='bold')
ax.legend(title='Condition', frameon=True)
plt.tight_layout()
plt.savefig('english_results.png', dpi=200, bbox_inches='tight')
plt.show()
print('📊 Chart saved as english_results.png')

In [ ]:
# STEP 10: Save results
df_mbert['model']  = 'mBERT'
df_base['model']   = 'XLM-RoBERTa-base'
df_large['model']  = 'XLM-RoBERTa-large'
df_xlmv['model']   = 'XLM-V-base'

df_all = pd.concat([df_mbert, df_base, df_large, df_xlmv])
df_all['language'] = 'English'
df_all.to_csv('english_results_detailed.csv', index=False)

summary = pd.DataFrame({
    'Model':        ['Children (Uzbek)', 'mBERT', 'XLM-RoBERTa-base', 'XLM-RoBERTa-large', 'XLM-V-base'],
    'Language':     ['Uzbek', 'English', 'English', 'English', 'English'],
    'Parameters':   ['N/A', '180M', '270M', '550M', '1.2B'],
    'ES_Pref_%':    [child_es, mbert_es, base_es, large_es, xlmv_es],
    'EO_Pref_%':    [child_eo, mbert_eo, base_eo, large_eo, xlmv_eo],
    'Difference_pp':[child_eo-child_es, mbert_eo-mbert_es, base_eo-base_es, large_eo-large_es, xlmv_eo-xlmv_es]
})
summary.to_csv('english_results_summary.csv', index=False)

from google.colab import files
files.download('english_results_summary.csv')
files.download('english_results_detailed.csv')
files.download('english_results.png')
print('\n✅ ALL DONE! Results downloaded.')